In [ ]:
# 1/3 - clone CUDA backend; DUNG .cu DA DO COMPILER SINH (khong chay .exe tren Linux)
import subprocess, os
print('===== DEBUG MEAN NOTEBOOK v2 - CORRECT FILE =====')
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
ex=d+'/Release/cuda_backend/examples'
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
cu=ex+'/debug_mean.cuda.cu'
print('cu exists', os.path.exists(cu))
if os.path.exists(cu):
    s=open(cu).read()
    print('has ltid threadIdx =', 'int t = threadIdx.x' in s)
    print('has shared =', '__shared__' in s)
    print(s[:900])

In [ ]:
# 2/3 - build .so
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/debug_mean.cuda.cu'; so=ex+'/debug_mean.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
p=subprocess.run(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+cu+' -o '+so+' -lcudart', shell=True, capture_output=True, text=True, timeout=600)
print('build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])

In [ ]:
# 3/3 - dump mean vs torch
import ctypes, torch
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
lib=ctypes.CDLL(ex+'/debug_mean.so'); I32=ctypes.c_int; I64=ctypes.c_int64; F32=ctypes.c_float
fn=lib.tkv_launch_debug_mean; fn.argtypes=[I64,I64,I32,I32,I32,I32]; fn.restype=I32
h=256; rows=8
x=torch.randn(rows,h,device='cuda',dtype=torch.float32)
out=torch.zeros(rows,device='cuda',dtype=torch.float32)
rc=fn(x.data_ptr(),out.data_ptr(),rows,h,rows,256)
print('rc=',rc)
print('TKV mean:',out.cpu().numpy())
print('TORCH mean:',x.mean(dim=1).cpu().numpy())
print('maxerr=',(out-x.mean(dim=1)).abs().max().item())